# ENTRENAMIENTO DE MODELOS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn import metrics
import xgboost
from sklearn import svm, datasets
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

# Análisis exploratorio

In [41]:
df_pasta_train = pd.read_csv("data/train.csv", sep=",")
df_pasta_train.head()

,ID,RevolvingUtilizationOfUnsecuredLines,Age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,9580,0.668999,58,2,0.449504,3425.0,9,1,1,1,1.0,0
1,39755,0.015922,71,0,6.000000,NaN,5,0,0,0,0.0,0
2,118799,0.183062,52,1,0.035593,5000.0,9,0,0,0,0.0,0
3,16489,0.162301,77,0,0.227886,2000.0,8,0,0,0,0.0,0
4,149857,0.404199,30,0,0.026010,5843.0,4,0,0,0,0.0,0


In [25]:
df_pasta_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105000 entries, 0 to 104999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   ID                                    105000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  105000 non-null  float64
 2   Age                                   105000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  105000 non-null  int64  
 4   DebtRatio                             105000 non-null  float64
 5   MonthlyIncome                         84164 non-null   float64
 6   NumberOfOpenCreditLinesAndLoans       105000 non-null  int64  
 7   NumberOfTimes90DaysLate               105000 non-null  int64  
 8   NumberRealEstateLoansOrLines          105000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  105000 non-null  int64  
 10  NumberOfDependents                    102236 non-null  float64
 11  

In [26]:
df_pasta_train["MonthlyIncome"].isnull().sum()

np.int64(20836)

In [27]:
df_pasta_train[["MonthlyIncome"]]

,MonthlyIncome
0,3425.0
1,NaN
2,5000.0
3,2000.0
4,5843.0
...,...
104995,9300.0
104996,5429.0
104997,3016.0
104998,14166.0


In [42]:
df_pasta_train["MonthlyIncome_sin_con"] = df_pasta_train["MonthlyIncome"].isnull().astype(int)

In [43]:
df_pasta_train

,ID,RevolvingUtilizationOfUnsecuredLines,Age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs,MonthlyIncome_sin_con
0,9580,0.668999,58,2,0.449504,3425.0,9,1,1,1,1.0,0,0
1,39755,0.015922,71,0,6.000000,NaN,5,0,0,0,0.0,0,1
2,118799,0.183062,52,1,0.035593,5000.0,9,0,0,0,0.0,0,0
3,16489,0.162301,77,0,0.227886,2000.0,8,0,0,0,0.0,0,0
4,149857,0.404199,30,0,0.026010,5843.0,4,0,0,0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
104995,79500,0.842886,33,0,0.182561,9300.0,10,0,0,0,1.0,0,0
104996,84928,0.805186,68,0,0.229466,5429.0,7,0,0,0,0.0,0,0
104997,56301,0.811494,51,2,3.709314,3016.0,26,0,4,0,0.0,1,0
104998,41912,0.412590,62,1,0.173290,14166.0,7,1,1,0,0.0,0,0


In [55]:
df_pasta_train["SeriousDlqin2yrs"].value_counts(normalize=True)

SeriousDlqin2yrs
0    0.933486
1    0.066514
Name: proportion, dtype: float64

## Tratamiento de nulos en MonthlyIncome tanto en train como en test

In [44]:
mediana_ingresos_mensuales = df_pasta_train["MonthlyIncome"].median()
mediana_ingresos_mensuales

np.float64(5400.0)

In [45]:
df_pasta_train["MonthlyIncome"] = df_pasta_train["MonthlyIncome"].fillna(mediana_ingresos_mensuales)

In [46]:
df_pasta_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105000 entries, 0 to 104999
Data columns (total 13 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   ID                                    105000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  105000 non-null  float64
 2   Age                                   105000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  105000 non-null  int64  
 4   DebtRatio                             105000 non-null  float64
 5   MonthlyIncome                         105000 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       105000 non-null  int64  
 7   NumberOfTimes90DaysLate               105000 non-null  int64  
 8   NumberRealEstateLoansOrLines          105000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  105000 non-null  int64  
 10  NumberOfDependents                    102236 non-null  float64
 11  

In [47]:
df_pasta_test = pd.read_csv("data/test.csv", sep=",")
df_pasta_test.head()

,ID,RevolvingUtilizationOfUnsecuredLines,Age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,129460,1.000000,21,0,8.000000,NaN,0,0,0,0,NaN
1,134018,0.009878,38,0,0.229978,10500.0,10,0,1,0,1.0
2,86523,0.276836,70,0,1914.000000,NaN,23,0,1,0,0.0
3,138466,0.045413,75,0,452.000000,NaN,4,0,0,0,0.0
4,143905,0.000000,82,0,0.000000,NaN,5,0,0,0,0.0


In [48]:
df_pasta_test["MonthlyIncome_sin_con"] = df_pasta_test["MonthlyIncome"].isnull().astype(int)

In [49]:
df_pasta_test["MonthlyIncome"] = df_pasta_test["MonthlyIncome"].fillna(mediana_ingresos_mensuales)

In [50]:
df_pasta_test

,ID,RevolvingUtilizationOfUnsecuredLines,Age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,MonthlyIncome_sin_con
0,129460,1.000000,21,0,8.000000,5400.0,0,0,0,0,NaN,1
1,134018,0.009878,38,0,0.229978,10500.0,10,0,1,0,1.0,0
2,86523,0.276836,70,0,1914.000000,5400.0,23,0,1,0,0.0,1
3,138466,0.045413,75,0,452.000000,5400.0,4,0,0,0,0.0,1
4,143905,0.000000,82,0,0.000000,5400.0,5,0,0,0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
44995,124596,0.055997,70,0,51.000000,5400.0,2,0,0,0,0.0,1
44996,75895,1.000000,62,0,2796.000000,5400.0,5,0,1,0,3.0,1
44997,92453,0.673065,56,1,0.511132,7500.0,9,0,2,0,4.0,0
44998,139288,1.000000,22,0,0.000000,2500.0,0,0,0,0,0.0,0


## Tratamiento de nulos en NumberOfDependents tanto en train como en test

In [51]:
df_pasta_train["NumberOfDependents_vacio_si_no"] = df_pasta_train["NumberOfDependents"].isnull().astype(int)
df_pasta_test["NumberOfDependents_vacio_si_no"] = df_pasta_test["NumberOfDependents"].isnull().astype(int)

In [52]:
df_pasta_train["NumberOfDependents"] = df_pasta_train["NumberOfDependents"].fillna(0)
df_pasta_test["NumberOfDependents"] = df_pasta_test["NumberOfDependents"].fillna(0)

In [53]:
df_pasta_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105000 entries, 0 to 104999
Data columns (total 14 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   ID                                    105000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  105000 non-null  float64
 2   Age                                   105000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  105000 non-null  int64  
 4   DebtRatio                             105000 non-null  float64
 5   MonthlyIncome                         105000 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       105000 non-null  int64  
 7   NumberOfTimes90DaysLate               105000 non-null  int64  
 8   NumberRealEstateLoansOrLines          105000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  105000 non-null  int64  
 10  NumberOfDependents                    105000 non-null  float64
 11  

In [54]:
df_pasta_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ID                                    45000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  45000 non-null  float64
 2   Age                                   45000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  45000 non-null  int64  
 4   DebtRatio                             45000 non-null  float64
 5   MonthlyIncome                         45000 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       45000 non-null  int64  
 7   NumberOfTimes90DaysLate               45000 non-null  int64  
 8   NumberRealEstateLoansOrLines          45000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  45000 non-null  int64  
 10  NumberOfDependents                    45000 non-null  float64
 11  MonthlyIncome_s

## Exportación de nuevos datasets

In [56]:
df_pasta_train.to_csv("data/train_limpio.csv", index=False)

In [57]:
df_pasta_test.to_csv("data/test_limpio.csv", index=False)